# Mat & Comp – ML 1: Datos con pandas
## Antes de que una máquina aprenda, hay que mirar los datos

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** Secciones 1 y 2

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar qué significa que una máquina **aprenda de datos**.
2. Cargar una tabla de datos (un archivo CSV) en un **DataFrame** de `pandas`.
3. Explorar una tabla: tamaño, columnas, promedios y resumen estadístico.
4. **Filtrar**, ordenar, crear columnas y agrupar datos.
5. Encontrar y tratar **datos faltantes**.
6. Graficar los datos y medir relaciones con la **correlación**.

## 🔍 ¿Estudiar más sube la nota?

Encuestamos a **60 estudiantes** de grados 8 a 11 sobre una semana normal:
- ¿cuántas horas estudiaron matemáticas fuera de clase?
- ¿cuántas horas durmieron por noche?
- ¿cuántas horas al día pasaron en redes sociales?
- ¿practican algún deporte?

y anotamos su **nota de matemáticas** (de 1.0 a 5.0).

✏️ *Antes de mirar los datos, escribe tus predicciones:*
1. *¿Qué influye más en la nota: las horas de estudio, el sueño o las redes sociales?*
2. *¿A los deportistas les va mejor, peor o igual?*

**Mis predicciones:** 

*(Los datos son ficticios pero realistas: los creamos para este curso con un modelo parecido a lo que muestran los estudios reales).*

## 1. ¿Qué es aprender de datos?

En la programación que hemos hecho hasta ahora, **nosotros escribimos las reglas**: "si la nota es mayor o igual a 3.0, aprobó".

En el **aprendizaje automático** (*machine learning*) es al revés: le damos al computador **muchos ejemplos** y él **encuentra las reglas** solo.

| Programación tradicional | Machine learning |
|---|---|
| reglas + datos → respuestas | datos + respuestas → reglas |
| "si el correo dice *GANASTE*, es spam" | se le muestran 10 000 correos marcados como spam o no, y aprende a reconocerlos |

Ejemplos que usas a diario: las recomendaciones de videos, el desbloqueo con la cara, el traductor y los asistentes de IA.

Todo empieza con **datos organizados en tablas**: cada **fila** es un ejemplo (un estudiante) y cada **columna** es una característica (horas de estudio, nota…). Por eso el primer paso siempre es aprender a manejar tablas. En Python, la herramienta para eso es **pandas**.

## 2. Una tabla hecha a mano

Con lo que ya sabes de Python, una tabla se puede guardar como un **diccionario de listas**: cada columna es una lista.

In [ ]:
tabla = {
    "nombre": ["Ana", "Luis", "Sara", "Juan", "Eva"],
    "horas_estudio": [8, 2, 11, 5, 7],
    "nota": [3.8, 2.4, 4.5, 3.0, 3.6],
}

# El promedio de la nota, a mano, como en la Sección 1:
notas = tabla["nota"]
print("Promedio a mano:", sum(notas) / len(notas))

Funciona, pero imagina hacerlo con 60 estudiantes y 7 columnas, o con un millón de filas. `pandas` convierte esa estructura en un **DataFrame**: una tabla con superpoderes.

In [ ]:
import pandas as pd          # "pd" es el apodo que todo el mundo usa

mini = pd.DataFrame(tabla)
mini

In [ ]:
# Ahora el promedio es una sola instrucción
print("Promedio con pandas:", mini["nota"].mean())

💡 Si la **última línea** de una celda es una variable (como `mini`), Colab la muestra como una tabla bonita sin necesidad de `print`.

## 3. Cargar los datos de los 60 estudiantes

Los datos reales casi siempre vienen en archivos **CSV** (*comma-separated values*): texto plano donde cada línea es una fila y las columnas van separadas por comas. Así se ven las primeras líneas de nuestro archivo:

```
estudiante,grado,horas_estudio,horas_sueno,horas_redes,deporte,nota_matematicas
E01,8,12.0,7.0,5.0,no,3.4
E02,10,5.5,5.5,1.0,sí,2.4
```

`pd.read_csv` lo lee directamente desde internet.

In [ ]:
url = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
df = pd.read_csv(url)
df.head()        # las primeras 5 filas

In [ ]:
print("Filas y columnas:", df.shape)
print("Columnas:", list(df.columns))

In [ ]:
# info() muestra el tipo de cada columna y cuántos valores NO están vacíos
df.info()

Fíjate: casi todas las columnas tienen **60** valores, pero `horas_sueno` tiene solo **57**. Faltan 3 datos. Volveremos a eso en la sección 7.

Los tipos `int64` y `float64` son números enteros y decimales; `object` o `str` son textos.

## 4. Mirar una columna

Una columna se saca con corchetes, igual que en un diccionario. El resultado se llama **Series**.

In [ ]:
notas = df["nota_matematicas"]
print("Promedio:", notas.mean())
print("Mínima:  ", notas.min())
print("Máxima:  ", notas.max())
print("Mediana: ", notas.median())

### Lo mismo, a mano con numpy

Detrás de pandas está **numpy**. `.to_numpy()` convierte la columna en un arreglo de numpy, y con él podemos calcular el promedio con la fórmula: sumar todo y dividir entre la cantidad.

In [ ]:
import numpy as np

x = notas.to_numpy()
print("Promedio a mano:", x.sum() / len(x))
print("Desviación estándar a mano:", np.sqrt(((x - x.mean()) ** 2).mean()))
print("Desviación estándar con pandas:", notas.std(ddof=0))

La **desviación estándar** mide qué tan dispersos están los datos alrededor del promedio: es (más o menos) la distancia típica de un dato al promedio.

`describe()` calcula todo eso para cada columna numérica de una vez:

In [ ]:
df.describe().round(2)

- `count`: cuántos datos hay · `mean`: promedio · `std`: desviación estándar
- `min` y `max`: el menor y el mayor
- `25%`, `50%`, `75%`: los **cuartiles**. El 50 % es la mediana: la mitad de los datos está por debajo.

## 5. Filtrar y ordenar

Para quedarse solo con algunas filas se escribe una **condición** entre corchetes. Es como un `if` que se aplica a todas las filas a la vez.

In [ ]:
# ¿Qué estudiantes estudian más de 12 horas a la semana?
df[df["horas_estudio"] > 12]

In [ ]:
# Varias condiciones: & significa "y", | significa "o". Cada condición va entre paréntesis.
mucho_estudio_pocas_redes = df[(df["horas_estudio"] > 10) & (df["horas_redes"] < 2)]
print("Cuántos son:", len(mucho_estudio_pocas_redes))
print("Su nota promedio:", mucho_estudio_pocas_redes["nota_matematicas"].mean().round(2))

In [ ]:
# Los 5 estudiantes con mejor nota
df.sort_values("nota_matematicas", ascending=False).head(5)

In [ ]:
# value_counts cuenta cuántas veces aparece cada valor
df["deporte"].value_counts()

## 6. Crear columnas y agrupar

Se crea una columna nueva asignándole una operación sobre otras columnas. La operación se hace **fila por fila automáticamente**, sin escribir un `for`.

In [ ]:
df["aprobo"] = df["nota_matematicas"] >= 3.0
df["horas_redes_semana"] = df["horas_redes"] * 7
df[["estudiante", "nota_matematicas", "aprobo", "horas_redes", "horas_redes_semana"]].head()

In [ ]:
print("Porcentaje que aprobó:", round(df["aprobo"].mean() * 100, 1), "%")

💡 El promedio de una columna de `True`/`False` es la **proporción** de `True`, porque `True` vale 1 y `False` vale 0 (lo viste en la Lección 2 de Python).

**`groupby`** separa la tabla en grupos y calcula algo para cada uno. Así respondemos la pregunta de los deportistas:

In [ ]:
df.groupby("deporte")["nota_matematicas"].mean().round(2)

In [ ]:
df.groupby("grado")[["horas_estudio", "horas_redes", "nota_matematicas"]].mean().round(2)

## 7. Datos faltantes

En la vida real siempre faltan datos: alguien no respondió, un sensor falló… pandas marca un dato faltante como `NaN` (*Not a Number*).

In [ ]:
print(df.isna().sum())          # cuántos faltan en cada columna
df[df["horas_sueno"].isna()]    # ¿quiénes son?

Hay dos soluciones comunes:
1. **Eliminar** las filas incompletas con `dropna()`. Es sencillo, pero se pierde información.
2. **Rellenar** el dato con un valor razonable, como el promedio, usando `fillna()`.

Ninguna es perfecta. Muchos modelos de machine learning **no funcionan** si hay `NaN`, así que siempre hay que decidir qué hacer.

In [ ]:
sin_faltantes = df.dropna()
print("Filas después de dropna:", len(sin_faltantes))

promedio_sueno = df["horas_sueno"].mean()
df["horas_sueno"] = df["horas_sueno"].fillna(promedio_sueno)
print("Faltantes después de fillna:", df["horas_sueno"].isna().sum())
print(f"(se rellenaron con el promedio: {promedio_sueno:.2f} horas)")

## 8. Graficar los datos

Un **histograma** muestra cómo se reparten los valores de una columna: cuántos estudiantes hay en cada rango de notas.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df["nota_matematicas"], bins=10, color="#5b8def", edgecolor="white")
plt.axvline(3.0, color="red", linestyle="--", label="nota para aprobar")
plt.xlabel("nota de matemáticas")
plt.ylabel("número de estudiantes")
plt.title("¿Cómo se reparten las notas?")
plt.legend()
plt.show()

Un **diagrama de dispersión** pone un punto por estudiante. Sirve para ver si dos columnas están relacionadas.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
columnas = ["horas_estudio", "horas_sueno", "horas_redes"]
titulos = ["Horas de estudio (semana)", "Horas de sueño (noche)", "Horas en redes (día)"]

for eje, col, titulo in zip(ejes, columnas, titulos):
    eje.scatter(df[col], df["nota_matematicas"], alpha=0.7)
    eje.set_xlabel(titulo)
    eje.grid(alpha=0.3)
ejes[0].set_ylabel("nota de matemáticas")
plt.suptitle("Cada punto es un estudiante")
plt.show()

¿Qué ves? Con las horas de estudio los puntos forman casi una **línea que sube**. Con las redes sociales, una nube que **baja** un poco. Con el sueño, la relación es más débil.

## 9. La correlación: un número para la relación

La **correlación** es un número entre **−1 y 1** que mide qué tan bien se alinean los puntos en una recta:

| Correlación | Significa |
|---|---|
| cerca de **1** | cuando una sube, la otra sube (línea que sube) |
| cerca de **0** | no hay relación lineal (nube sin forma) |
| cerca de **−1** | cuando una sube, la otra baja (línea que baja) |

In [ ]:
numericas = ["horas_estudio", "horas_sueno", "horas_redes", "nota_matematicas"]
correlaciones = df[numericas].corr().round(2)
correlaciones

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(correlaciones, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="correlación")
plt.xticks(range(4), numericas, rotation=30, ha="right")
plt.yticks(range(4), numericas)
for i in range(4):
    for j in range(4):
        plt.text(j, i, correlaciones.iloc[i, j], ha="center", va="center")
plt.title("Mapa de correlaciones")
plt.show()

⚠️ **Correlación no es causalidad.** Que dos cosas vayan juntas no prueba que una cause la otra. Por ejemplo, en verano se venden más helados y también hay más ahogados en piscinas, pero los helados no causan ahogamientos: ambos dependen del calor.

¿Se cumplieron tus predicciones del inicio?

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …escribes `df["nota"]` en vez de `df["nota_matematicas"]`? ¿Qué error aparece?
2. …filtras con `and` en vez de `&`: `df[(df["grado"] == 11) and (df["deporte"] == "sí")]`?
3. …cambias `bins=10` por `bins=3` o `bins=40` en el histograma? ¿Cuál cuenta mejor la historia?
4. …calculas `df["horas_estudio"].corr(df["horas_estudio"])`? ¿Por qué da ese valor?
5. …usas `df.groupby("aprobo")["horas_redes"].mean()`? ¿Los que aprueban usan más o menos redes?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil).** ¿Cuántos estudiantes hay de cada grado? ¿Cuál es el promedio de horas de sueño de toda la tabla?

**Ejercicio 2 (fácil).** Compara la nota promedio de los que estudian **más de 10 horas** con la de los que estudian **menos de 3 horas**. ¿Cuánta diferencia hay?

**Ejercicio 3 (medio).** Crea una columna `redes_mucho` que sea `True` si el estudiante pasa **4 horas o más** al día en redes. Usa `groupby` para calcular el **porcentaje de aprobados** en cada grupo.

**Ejercicio 4 (medio).** Haz un diagrama de dispersión de `horas_redes` contra `nota_matematicas`, con los deportistas de un color y los no deportistas de otro. Pista: filtra dos tablas y llama a `plt.scatter` dos veces, con `label=` y `plt.legend()`.

**Ejercicio 5 (medio).** Crea una columna `bienestar = horas_sueno - horas_redes` y calcula su correlación con la nota. ¿Es mayor o menor que la correlación del sueño solo y de las redes solas? ¿Qué crees que significa?

## 🏆 Reto opcional: casas de California

Colab trae algunos conjuntos de datos reales en la carpeta `sample_data` (ábrela en el panel de la izquierda 📁). Uno de ellos describe casas en distintas zonas de California en 1990.

1. Cárgalo con `pd.read_csv("sample_data/california_housing_train.csv")` y mira cuántas filas y columnas tiene.
2. Usa `describe()` para encontrar el precio mediano más alto (`median_house_value`).
3. ¿Qué columna tiene la correlación más alta con `median_house_value`?
4. Haz un diagrama de dispersión de `longitude` contra `latitude`, coloreado por el precio (`c=df_casas["median_house_value"]`). ¿Reconoces la forma de California y dónde están las casas caras?

## 🎬 Cierre

Hoy aprendiste que:
- En machine learning el computador **encuentra reglas a partir de ejemplos**, y todo empieza con datos en tablas.
- Un **DataFrame** de pandas se carga con `pd.read_csv`, y se explora con `head`, `shape`, `info` y `describe`.
- Se puede **filtrar** con condiciones, **ordenar**, **crear columnas** y **agrupar** con `groupby`, sin escribir ciclos.
- Los **datos faltantes** se eliminan o se rellenan, pero hay que decidirlo.
- La **correlación** mide qué tan alineados están dos conjuntos de datos, pero **no prueba causalidad**.

**Para pensar:** las horas de estudio y la nota tienen una correlación cercana a 0.9: los puntos casi forman una recta. Si un estudiante nuevo nos dice que estudia 9 horas a la semana, ¿podríamos **predecir** su nota? ¿Cuál es la **mejor recta** que pasa por esa nube de puntos? Esa es la pregunta de la próxima lección: la **regresión lineal**, el primer modelo de machine learning. 🚀